## DX702 Coding Homework ##
Keshia-Lee Martin<br>
October 11, 2026

### Week 1

In [22]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew

hw1_1 = pd.read_csv("homework_1.1.csv")
hw1_2 = pd.read_csv("homework_1.2.csv")

Question 1: Which of the following is closest to the coefficient of X1?

In [23]:
model = smf.ols("Y ~ X1 + X2 + X3", data=hw1_1).fit()
coeff_x1 = model.params["X1"]
print(round(coeff_x1, 3))

1.007


Question 2: Which Xi has the greatest difference between the amount Y increases for each 1 unit of Xi (fixing the other Xi's), as opposed to the amount that Y increases for each 1 unit of Xi in the dataset on average (not fixing the other Xi's)?
Hint: For the former, regress Y on Xi alone. For the latter, regress Y on all three Xi's.


In [24]:
only_x1   = smf.ols("Y ~ X1", data=hw1_1).fit()
only_x2   = smf.ols("Y ~ X2", data=hw1_1).fit()
only_x3   = smf.ols("Y ~ X3", data=hw1_1).fit()

coeff_x1 = model.params["X1"]
coeff_x2 = model.params["X2"]
coeff_x3 = model.params["X3"]

coeff_x1_alone = only_x1.params["X1"]
coeff_x2_alone = only_x2.params["X2"]
coeff_x3_alone = only_x3.params["X3"]

print("X1:", round(coeff_x1, 3), " alone", round(coeff_x1_alone, 3),
      " diff", round(coeff_x1_alone - coeff_x1, 3))
print("X2:", round(coeff_x2, 3), " alone", round(coeff_x2_alone, 3),
      " diff", round(coeff_x2_alone - coeff_x2, 3))
print("X3:", round(coeff_x3, 3), " alone", round(coeff_x3_alone, 3),
      " diff", round(coeff_x3_alone - coeff_x3, 3))

X1: 1.007  alone 1.842  diff 0.835
X2: 1.965  alone 4.084  diff 2.119
X3: 2.975  alone 3.097  diff 0.122


Question 3: When regressing Y on all Xi's together, which coefficient is most significant, taking the t-statistic as the measure of significance?

In [25]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Sun, 11 Oct 2026   Prob (F-statistic):               0.00
Time:                        18:20:12   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0026      0.016      0.166      0.8

Question 4: What is the distance of the farthest match in this set?

In [26]:
treated = hw1_2[hw1_2["X"] == 1].reset_index(drop=True)
control = hw1_2[hw1_2["X"] == 0].reset_index(drop=True)

n_neighbor = NearestNeighbors(n_neighbors=1).fit(control[["Z"]])
distances, positions = n_neighbor.kneighbors(treated[["Z"]])
 
print("Treated units =", len(treated))
print("Farthest match distance =", distances.max())


Treated units = 48
Farthest match distance = 0.2102170871093757


Question 5: What is the effect? The difference between the average Y for the X = 0 values and the average Y for the X = 1 values, where the X = 0 sample is the set of best matches — so the matched sample of X = 0 against the full sample of X = 1.

In [27]:
matched = control.iloc[positions.flatten()]

print("mean Y treated =", treated["Y"].mean())
print("mean Y matched control =", matched["Y"].mean())
print("Effect =", treated["Y"].mean() - matched["Y"].mean())


mean Y treated = 1.125597137873841
mean Y matched control = 0.582237072655257
Effect = 0.5433600652185839


Question 6: How many duplicates do you end up with? Count all but the first in each group. One way to do this is radius_neighbors.

In [28]:
pairs_total = 0
controls_used = set()

n_neighbor2 = NearestNeighbors(radius=0.2).fit(control[["Z"]])
distances, groups = n_neighbor2.radius_neighbors(treated[["Z"]])

for group in groups:
    pairs_total = pairs_total + len(group)
    for position in group:
        controls_used.add(int(position))

print(pairs_total)
print(len(controls_used))
print(pairs_total - len(controls_used))

737
52
685


Question 7: What is the effect?


In [29]:
averages = []
for group in groups:
    if len(group) > 0:
        averages.append(control.iloc[group]["Y"].mean())
 
overall_control = sum(averages) / len(averages)
 
print(len(averages))
print(overall_control)
print(treated["Y"].mean() - overall_control)


46
0.5411846604492228
0.5844124774246182


### Week 2

Which of these is closest to being the coefficient of group 1? 

In [30]:
hw2_1 = pd.read_csv("homework_2.1.csv")
hw2_2 = pd.read_csv("homework_2.2.csv")


In [31]:
model_g1 = smf.ols("G1 ~ time", data=hw2_1).fit()
model_g2 = smf.ols("G2 ~ time", data=hw2_1).fit()
model_g3 = smf.ols("G3 ~ time", data=hw2_1).fit()
 
slope_g1 = model_g1.params["time"]
slope_g2 = model_g2.params["time"]
slope_g3 = model_g3.params["time"]
 
print("G1 =", slope_g1)
print("G2 =", slope_g2)
print("G3 =", slope_g3)


G1 = 0.008498349168750096
G2 = 0.0070804680232676815
G3 = 0.011472822938047054


Which of these is closest to being the common linear coefficient for all groups?

In [32]:
g1_rows = hw2_1[["time", "G1"]].rename(columns={"G1": "y"})
g2_rows = hw2_1[["time", "G2"]].rename(columns={"G2": "y"})
g3_rows = hw2_1[["time", "G3"]].rename(columns={"G3": "y"})

g1_rows["group"] = "G1"
g2_rows["group"] = "G2"
g3_rows["group"] = "G3"

all_rows = pd.concat([g1_rows, g2_rows, g3_rows])

all_rows["is_g1"] = (all_rows["group"] == "G1").astype(int)
all_rows["is_g2"] = (all_rows["group"] == "G2").astype(int)
all_rows["is_g3"] = (all_rows["group"] == "G3").astype(int)

pooled = smf.ols("y ~ is_g1 + is_g2 + is_g3 + time - 1", data=all_rows).fit()

fixed_effect_g1 = pooled.params["is_g1"]
fixed_effect_g2 = pooled.params["is_g2"]
fixed_effect_g3 = pooled.params["is_g3"]
common_slope = pooled.params["time"]

print("G1 fixed effect =", fixed_effect_g1)
print("G2 fixed effect =", fixed_effect_g2)
print("G3 fixed effect =", fixed_effect_g3)
print("common slope, time =", common_slope)


G1 fixed effect = 0.07855194562167919
G2 fixed effect = 0.5896544232284033
G3 fixed effect = 0.26903180757269846
common slope, time = 0.009017213376688394


If we were to measure the effect of the treatment simply by subtracting the outcomes of the treated vs. untreated population, which of these is closest to the mean effect? (This is not the recommended way of measuring the mean effect when there are confounders!) 

In [33]:
treated_y = hw2_2[hw2_2["X"] == 1]["Y"]
control_y = hw2_2[hw2_2["X"] == 0]["Y"]
 
mean_treated = treated_y.mean()
mean_control = control_y.mean()
naive_effect = mean_treated - mean_control
 
print("n treated =", len(treated_y), " n control =", len(control_y))
print("mean Y, treated =", mean_treated)
print("mean Y, control =", mean_control)
print("naive effect =", naive_effect)


n treated = 1972  n control = 8028
mean Y, treated = 7.842756767343214
mean Y, control = 4.922039502620024
naive effect = 2.9207172647231907


If we were to use bootstrap sampling to measure the variance of that effect, again finding the effect using the non-recommended approach, which of these is closest to that variance?

In [34]:
effects = []

for i in range(10000):
    sample = hw2_2.sample(n=len(hw2_2), replace=True)
 
    mean_treated = sample[sample["X"] == 1]["Y"].mean()
    mean_control = sample[sample["X"] == 0]["Y"].mean()
 
    effects.append(mean_treated - mean_control)
 
bootstrap_var = np.var(effects, ddof=1)
print("bootstrap variance is", bootstrap_var)
 
# Check it against the textbook formula for a difference of two means.
treated_y = hw2_2[hw2_2["X"] == 1]["Y"]
control_y = hw2_2[hw2_2["X"] == 0]["Y"]
 
formula_var = treated_y.var() / len(treated_y) + control_y.var() / len(control_y)
print("formula variance is", round(formula_var, 5))


bootstrap variance is 0.032339705837420296
formula variance is 0.03188


if we ran a linear regression (with intercept) to measure the effect, which of these is closest to the skewness of the effect measured? (Look up skewness online. You can use scipy.stats.skew to compute the skewness of a list of numbers.) 

In [35]:
effects = []

for i in range(10000):
    sample = hw2_2.sample(n=len(hw2_2), replace=True)
 
    fit = smf.ols("Y ~ X + Z", data=sample).fit()
    effect_this_sample = fit.params["X"]
 
    effects.append(effect_this_sample)
 
print("mean effect is", np.mean(effects))
print("skewness is", skew(effects))


mean effect is 2.8178816317857787
skewness is 0.06456188760114985
